In [1]:
# TRAIN MODELS AND GENERATE WEIGHTS #

import os
import numpy as np
import pandas as pd
import xarray as xr
import joblib
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from models import SimpleTabularModel, estimator_map
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def _prefix(d, pre='est__'):
    return {pre + k: v for k, v in d.items()}

# Configuration paths
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "clean_pfisr_data.nc"
WGTS_DIR = BASE_DIR / "ckpts" / "checkpoints_1"
WGTS_DIR.mkdir(parents=True, exist_ok=True)

# Load and format data
ds = xr.open_dataset(DATA_PATH)
df_full = ds.to_dataframe().reset_index()
df_full['year'] = df_full['time'].dt.year

for xx in range(1):
    for year in range(2008, 2009):
        
        target_alts = sorted(df_full['GDALT'].unique(), reverse=True)
        
        for alt in target_alts:
            if alt > 400 or alt < 100:
                continue
                
            local_times = sorted(df_full['SLT'].unique())
            
            for lt in local_times:
                # Isolate data for specific altitude and local time
                df_bin = df_full[(df_full['SLT'] == lt) & (df_full['GDALT'] == alt)].copy()
                
                # Apply error thresholding and unit conversion
                df_bin = df_bin[df_bin['DNE'] < df_bin['NE']]
                df_bin['NE'] /= 1e9
                df_bin['DNE'] /= 1e9
                
                # Remove extreme outliers
                df_bin = df_bin[df_bin['NE'] > df_bin['DNE']]
                df_bin = df_bin[df_bin['NE'] < df_bin['NE'].mean() + (df_bin['NE'].std() * 10)]
                
                # Inject measurement noise into target variable
                rng = np.random.default_rng()
                df_bin['NE'] = rng.normal(loc=df_bin['NE'].to_numpy(), scale=df_bin['DNE'].to_numpy())
                df_bin['NE'] = df_bin['NE'].clip(lower=0.0)
                
                # Format time features and filter out validation year
                df_bin = df_bin.sort_values('time')
                df_bin['doy'] = df_bin['time'].dt.dayofyear
                df_bin = df_bin[df_bin['year'] != year]

                feature_cols = ['ap', 'F107', 'doy']
                df_bin = df_bin[['NE'] + feature_cols].dropna()
                
                if len(df_bin) < 2000:
                    continue

                # The model dictionary defines our four core algorithms
                model_configs = [
                    ('rf', SimpleTabularModel(model_type='rf')),   # Random Forest: builds multiple decision trees and averages them
                    ('gb', SimpleTabularModel(model_type='gb')),   # Gradient Boosting: builds trees sequentially to fix errors of previous ones
                    ('mlp', SimpleTabularModel(model_type='mlp')), # Multilayer Perceptron: a basic neural network architecture
                    ('lr', SimpleTabularModel(model_type='lr'))    # Linear Regression: fits a standard straight line to the data
                ]

                for est_type, model in model_configs:
                    save_name = f"{est_type}_LT{lt}_alt{alt}_outyear{year}_iter{xx}.pkl"
                    save_path = os.path.join(WGTS_DIR, save_name)
                    
                    if os.path.exists(save_path):
                        continue
                        
                    print(f"Training {est_type.upper()} | LT: {lt} | Alt: {alt} | Year: {year}")

                    base_est, param_dist = estimator_map[est_type]
                    
                    # A Pipeline bundles data processing and modeling into a single object.
                    # Standard scaling forces all input variables to have a mean of 0 and variance of 1,
                    # which prevents variables with naturally large numbers from dominating the algorithm.
                    pipe = Pipeline([('scaler', StandardScaler()), ('est', base_est)])
                    pipe_param_dist = _prefix(param_dist, 'est__')
                    
                    X = df_bin[feature_cols].values
                    y = df_bin['NE'].values
                    
                    # TimeSeriesSplit breaks the dataset into chronological chunks. 
                    # This ensures the model is always evaluated on future data rather than randomly selected past data.
                    cv = TimeSeriesSplit(n_splits=5)
                    
                    n_iter_search = 20 if est_type == 'mlp' else 50 if est_type == 'rf' else 40
                    
                    # RandomizedSearchCV automates the hunt for the best model settings (hyperparameters).
                    # Instead of testing every possible combination, it samples random configurations n_iter times.
                    search = RandomizedSearchCV(
                        estimator=pipe,
                        param_distributions=pipe_param_dist,
                        n_iter=n_iter_search,
                        scoring='neg_root_mean_squared_error',
                        cv=cv,
                        n_jobs=-1, # Uses all available CPU cores simultaneously to speed up the search
                        random_state=42,
                        verbose=0
                    )
                    
                    # This triggers the actual training and evaluation loop across the different guessed configurations.
                    search.fit(X, y)
                    
                    # Extract the winning settings from the automated search object.
                    best_readable = {k.replace('est__',''): v for k, v in search.best_params_.items()}
                    
                    model.model_params = best_readable
                    if est_type == 'lr' and 'alpha' in model.model_params:
                        model.model_params.pop('alpha') 
                        
                    # Now that we know the optimal settings, we initialize a fresh model and fit it to our data.
                    model.fit(df_bin, validation_split=0.1)
                    
                    # Serialize and save the trained "brain" to disk so it can be loaded for predictions later.
                    joblib.dump(model, save_path)

Training MLP | LT: 0.0 | Alt: 400.0 | Year: 2008
Training LR | LT: 0.0 | Alt: 400.0 | Year: 2008
Training RF | LT: 1.0 | Alt: 400.0 | Year: 2008
Training GB | LT: 1.0 | Alt: 400.0 | Year: 2008
Training MLP | LT: 1.0 | Alt: 400.0 | Year: 2008


Process LokyProcess-16:
Traceback (most recent call last):
  File "/home/ben-martinez/.local/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py", line 490, in _process_worker
    r = call_item()
        ^^^^^^^^^^^
  File "/home/ben-martinez/.local/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py", line 291, in __call__
    return self.fn(*self.args, **self.kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ben-martinez/.local/lib/python3.12/site-packages/joblib/parallel.py", line 607, in __call__
    return [func(*args, **kwargs) for func, args, kwargs in self.items]
            ^^^^^^^^^^^^^^^^^^^^^
  File "/home/ben-martinez/.local/lib/python3.12/site-packages/sklearn/utils/parallel.py", line 184, in __call__
    return self.function(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ben-martinez/.local/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
   

  File "/home/ben-martinez/.local/lib/python3.12/site-packages/sklearn/utils/extmath.py", line 230, in safe_sparse_dot
    sparse.issparse(a)
  File "/home/ben-martinez/.local/lib/python3.12/site-packages/scipy/_lib/_sparse.py", line 10, in issparse
    def issparse(x):
    
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/ben-martinez/.local/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py", line 493, in _process_worker
    result_queue.put(_ResultItem(call_item.work_id, exception=exc))
  File "/home/ben-martinez/.local/lib/python3.12/site-packages/joblib/externals/loky/backend/queues.py", line 235, in put
    with self._wlock:
  File "/home/ben-martinez/.local/lib

In [ ]:
# LOAD TRAINED MODELS AND GENERATE PREDICTIONS #

import os
import numpy as np
import pandas as pd
import xarray as xr
import joblib
from scipy.stats import pearsonr
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "clean_pfisr_data.nc"
WGTS_DIR = BASE_DIR / "ckpts" / "checkpoints_1" 
OUT_CSV = BASE_DIR / "hold_it.csv"

ds = xr.open_dataset(DATA_PATH)
df_eval = ds.to_dataframe().reset_index()

features = ['ap', 'F107', 'doy']
model_types = ['gb', 'mlp', 'lr', 'rf']
target_alts = np.arange(100, 410, 10)
num_iterations = 100

compiled_results = []

# Running multiple iterations helps quantify the uncertainty and stability of the model's internal logic.
for iter_num in range(num_iterations):
    # We iterate through the years to perform strict out-of-sample validation.
    # The algorithm is evaluated exclusively on a year of data it has never seen during training.
    for year in range(2007, 2026):
        
        test_start = pd.to_datetime(f'{year}-01-01')
        test_end = test_start + pd.Timedelta(days=365)
        yearly_results = []
        
        print(f'Evaluating Year: {year} | Iteration: {iter_num}')
        
        for alt in target_alts:
            df_alt = df_eval[df_eval['GDALT'] == alt].copy()
            if df_alt.empty:
                continue

            test_data = df_alt[(df_alt['datetime'] >= test_start) & (df_alt['datetime'] <= test_end)].copy()
            if test_data.empty:
                continue
            
            for m_type in model_types:
                test_data[f"NE_pred_{m_type}"] = np.nan

                for slt in sorted(df_alt['SLT'].unique()):
                    test_slt = test_data[test_data['SLT'] == slt].copy()
                    
                    # Filter out highly anomalous peaks to prevent extreme, rare measurement spikes 
                    # from artificially skewing the baseline predictability metrics.
                    threshold = test_slt['NE'].mean() + (5 * test_slt['NE'].std())
                    test_slt = test_slt[test_slt['NE'] < threshold]
                    
                    if len(test_slt) < 10:
                        continue
                        
                    filename = f"{m_type}_LT{float(slt):.1f}_alt{float(alt):.1f}_outyear{year}_iter{iter_num}.pkl"
                    model_path = os.path.join(WGTS_DIR, filename)
                    
                    if not os.path.exists(model_path):
                        continue
        
                    # Rehydrate the saved model into memory.
                    model = joblib.load(model_path)
                    X_test = test_slt[features].copy()
                    y_test = test_slt['NE']
                    
                    # Generate base correlation metrics to establish the overall accuracy 
                    # before we start breaking things to test feature importance.
                    preds = model.predict(X_test)['NE_pred'].values
                    base_corr, _ = pearsonr(preds, y_test)
                    
                    # Permutation Importance: To figure out which physical drivers matter most, 
                    # we intentionally scramble one column of input data at a time while leaving the rest intact.
                    # If the model's accuracy suddenly crashes, we know that specific feature was critical.
                    importances = {}
                    for feat in features:
                        X_shuffled = X_test.copy()
                        X_shuffled[feat] = np.random.permutation(X_shuffled[feat].values)
                        shuff_preds = model.predict(X_shuffled)['NE_pred'].values
                        shuff_corr, _ = pearsonr(shuff_preds, y_test)
                        
                        # The drop in correlation represents the feature's importance.
                        importances[feat] = max(0, base_corr - shuff_corr)
        
                    # Calculate permutation importance for grouped inputs.
                    # Here we scramble solar flux and day-of-year together to evaluate the 
                    # combined influence of solar and seasonal cycles versus geomagnetic activity.
                    X_group = X_test.copy()
                    X_group['F107'] = np.random.permutation(X_group['F107'].values)
                    X_group['doy'] = np.random.permutation(X_group['doy'].values)
                    group_preds = model.predict(X_group)['NE_pred'].values
                    group_corr, _ = pearsonr(group_preds, y_test)
                    solar_seasonal_imp = max(0, base_corr - group_corr)
        
                    # Normalize the absolute importance scores into relative percentage contributions.
                    # This tells us exactly what fraction of the model's success comes from knowing 'ap' vs 'F107'.
                    total_imp = importances['ap'] + solar_seasonal_imp
                    geomag_contrib = (importances['ap'] / total_imp) * base_corr if total_imp > 0 else 0
                    solar_contrib = (solar_seasonal_imp / total_imp) * base_corr if total_imp > 0 else 0
        
                    # Store variables for output compilation
                    test_slt['NE_pred'] = preds
                    test_slt['model_type'] = m_type
                    test_slt['geomag_importance'] = importances['ap']
                    test_slt['solar_importance'] = solar_seasonal_imp
                    test_slt['geomag_contribution'] = geomag_contrib
                    test_slt['solar_contribution'] = solar_contrib
                    test_slt['total_correlation'] = base_corr
                    test_slt['iter'] = iter_num
                    
                    cols_to_keep = ['datetime', 'GDALT', 'SLT', 'NE', 'NE_pred', 'model_type', 
                                    'geomag_importance', 'solar_importance', 'geomag_contribution', 
                                    'solar_contribution', 'total_correlation', 'ap', 'F107', 'doy', 'iter']
                    
                    yearly_results.append(test_slt[cols_to_keep])
        
        if yearly_results:
            compiled_results.extend(yearly_results)

if compiled_results:
    df_final_output = pd.concat(compiled_results)
    df_final_output.to_csv(OUT_CSV, index=False)
    print(f"Processing complete. Data written to {OUT_CSV}.")
else:
    print("No predictions generated. Please verify file paths and weights directories.")